<a href="https://colab.research.google.com/github/VamuveTV/3DTrajMaster/blob/main/GugaMultipleVoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install SpeechBrain**

In [1]:
# Install SpeechBrain (for separation) and necessary audio utilities
!pip install --upgrade speechbrain
!pip install librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 28.0 MB/s eta 0:00:00


**Mount Google Drive**

In [2]:
from google.colab import drive
import os
import io
from pathlib import Path

drive.mount('/gdrive')

Mounted at /gdrive


**Define Configuration**

In [3]:
# Customize the following options!
# This is the path to the 2-speaker separation model (Sepformer).
separation_model_path = "speechbrain/sepformer-wsj02mix"

# File types to look for
extensions = ["mp3", "wav", "ogg", "flac"]

# Output options
mp3 = True        # Convert final output files to MP3? (Saves space)
mp3_rate = 320    # MP3 bitrate in kbps (320 is high quality)

# --- YOUR CUSTOM PATHS BELOW ---
# The folder where your audio files (e.g., meeting.mp3) are located.
in_path = '/gdrive/MyDrive/demucs/'

# The folder where the separated voice files will be saved.
out_path = '/gdrive/MyDrive/voice_separated/'

**Define Separation Functions**

In [4]:
#@title Useful functions, don't forget to execute
import subprocess as sp
import sys
import torch
import torchaudio
import soundfile as sf
from speechbrain.inference.separation import SepformerSeparation

def find_files(in_path):
    out = []
    for item in os.listdir(in_path):
        file = Path(in_path) / item
        if file.suffix.lower().lstrip(".") in extensions:
            out.append(file)
    return out

# Initialize the separation model once
try:
    # Load the Sepformer model for two-speaker separation
    device = "cuda" if torch.cuda.is_available() else "cpu"
    separator = SepformerSeparation.from_hparams(source=separation_model_path,
                                                 run_opts={"device":device})
    print(f"SpeechBrain Sepformer model loaded successfully on {device}.")
except Exception as e:
    print(f"Error loading SpeechBrain model: {e}")
    separator = None

def separate_voices(inp=None, outp=None):
    if separator is None:
        print("Cannot run separation: Model failed to load.")
        return

    inp = inp or in_path
    outp = outp or out_path
    Path(outp).mkdir(parents=True, exist_ok=True) # Ensure output path exists

    files_to_process = find_files(inp)
    if not files_to_process:
        print(f"No valid audio files in {inp}")
        return

    print("Going to separate the files:")
    print('\n'.join(map(str, files_to_process)))

    for file_path in files_to_process:
        print(f"\nProcessing: {file_path.name}")

        # 1. Load the audio file
        mix, rate = torchaudio.load(file_path)
        # Convert to mono if it's stereo
        if mix.ndim > 1:
            mix = mix.mean(dim=0).unsqueeze(0)

        # 2. Run the separation model
        try:
            est_sources = separator.separate_batch(mix.to(device))
            num_sources = est_sources.shape[0]

            # 3. Save the separated sources
            base_name = file_path.stem
            output_folder = Path(outp) / base_name
            output_folder.mkdir(parents=True, exist_ok=True)

            print(f"Found {num_sources} separated streams. Saving to: {output_folder}")

            for i in range(num_sources):
                # Save as high-quality WAV (intermediate step)
                wav_file = output_folder / f"speaker_{i+1}.wav"
                sf.write(str(wav_file), est_sources[i].squeeze().cpu().numpy(), rate)

                # 4. (Optional) Convert to MP3
                if mp3:
                    mp3_file = output_folder / f"speaker_{i+1}.mp3"
                    cmd = [
                        "ffmpeg",
                        "-i", str(wav_file),
                        "-b:a", f"{mp3_rate}k",
                        "-y",
                        str(mp3_file)
                    ]
                    sp.run(cmd, stdout=sp.PIPE, stderr=sp.PIPE)
                    print(f" - Saved speaker {i+1} as MP3: {mp3_file.name}")
                    os.remove(wav_file)
                else:
                    print(f" - Saved speaker {i+1} as WAV: {wav_file.name}")

        except Exception as e:
            print(f"Separation failed for {file_path.name}: {e}")

AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'

**Run Separation**

In [ ]:
# EXECUTE THIS TO START SEPARATION ON YOUR GOOGLE DRIVE FILES
separate_voices()